# Servidor Único de Inferencia + Agente Ollama LLM en Google Colab (1 Sola URL)

Levanta en GPU T4 de Google Colab **ambos servicios detrás de UNA SOLA URL HTTPS**:
1. **Inferencia Unificada (`/infer`, `/infer/placa`):** M1+OCR multi-ángulo, M2 (vehículos seg), M3 (1..6+ ejes), M4 (pasajeros estrictamente dentro de cabina, sin transeúntes).
2. **Agente LLM Propio (`/ollama/api/chat`):** Ollama con `llama3.2:3b` integrado por proxy interno (no requiere segundo túnel ni configurar `OLLAMA_URL` aparte).

Al ejecutar la Celda 2 copia la **única URL `https://....trycloudflare.com`** y pégala en **`INFERENCE_URL`** (en `https://peaje-ia-web.onrender.com` -> pestaña **Estado**). El backend detectará y activará **Inferencia + Ollama LLM automáticamente**.

In [ ]:
!rm -rf /content/proyecto_caseta_IA
!git clone https://github.com/yeixd/proyecto_caseta_IA.git /content/proyecto_caseta_IA
%cd /content/proyecto_caseta_IA/inferencia
!pip install -q fastapi uvicorn[standard] python-multipart pydantic httpx ultralytics easyocr opencv-python-headless python-dotenv
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!python diagnostico_entorno.py

In [ ]:
# (Opcional) Si ya tienes tus pesos entrenados .pt en Google Drive, descomenta estas 2 líneas:
# from google.colab import drive
# drive.mount('/content/drive')

import os, subprocess, time, re, urllib.request

OLLAMA_MODEL = 'llama3.2:3b'

os.environ['MODEL_M1_PLATES_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m1_placas.pt'
os.environ['MODEL_M2_VEHICLE_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m2_vehiculos_seg.pt'
os.environ['MODEL_M3_WHEEL_POSE_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m3_ruedas_pose.pt'
os.environ['MODEL_M4_PASSENGERS_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m4_pasajeros_seg.pt'
os.environ['INFERENCE_API_KEY'] = ''
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_ORIGINS'] = '*'

# 1. Iniciar Ollama y descargar modelo LLM
print('1/3 Iniciando Ollama...')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(15):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        break
    except Exception:
        time.sleep(1)
print(f'    Descargando modelo {OLLAMA_MODEL}...')
subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
print('    Ollama LISTO con', OLLAMA_MODEL)

# 2. Iniciar servidor único FastAPI (Inferencia + Proxy /ollama)
print('2/3 Iniciando servidor FastAPI (M1+M2+M3+M4 + Proxy Ollama) en puerto 8000...')
infer_proc = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'])
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
        break
    except Exception:
        time.sleep(2)

# Verificar que el proxy /ollama/api/tags responda OK
print('    Verificando proxy Ollama:', urllib.request.urlopen('http://127.0.0.1:8000/ollama/api/tags', timeout=5).read().decode()[:160])

# 3. Abrir UN SOLO túnel HTTPS seguro
print('3/3 Abriendo túnel HTTPS público...')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in tunnel.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        print('\n======================================================================')
        print('PEGA ESTA ÚNICA URL EN https://peaje-ia-web.onrender.com -> ESTADO:')
        print('  INFERENCE_URL :', public_url)
        print('  OLLAMA_URL    :', public_url + '/ollama')
        print('  Modelo Ollama :', OLLAMA_MODEL)
        print('======================================================================\n')
        break

print('Inferencia + Ollama LLM activos. Mantén esta celda ejecutándose.')
infer_proc.wait()